In [12]:
# Cell 1 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import zipfile, os
os.makedirs('/content/dataset/voc', exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/RD2/voc_dataset.zip', 'r') as z:
    z.extractall('/content/dataset/voc')
print("Done")

Done


In [ ]:
def show_tree(path, indent=0):
    for item in sorted(os.listdir(path)):
        print("  " * indent + item)
        full = os.path.join(path, item)
        if os.path.isdir(full) and indent < 4:
            show_tree(full, indent + 1)

show_tree('/content/dataset/voc')

voc
  .DS_Store
  Smartphone-Detection-2
    .DS_Store
    README.roboflow.txt
    train
      IMG_2379_jpg.rf.7365fa9ff9cd1ff3a4cd55745336221f.jpg
      IMG_2379_jpg.rf.7365fa9ff9cd1ff3a4cd55745336221f.xml
      IMG_2380_jpg.rf.26f17b5795f5a201b3b39b9994431298.jpg
      IMG_2380_jpg.rf.26f17b5795f5a201b3b39b9994431298.xml
      IMG_2381_jpg.rf.0b7a47d4e8a67a988f15c32629ef3b49.jpg
      IMG_2381_jpg.rf.0b7a47d4e8a67a988f15c32629ef3b49.xml
      IMG_2382_jpg.rf.8744377541452bc0e6a20045dea52682.jpg
      IMG_2382_jpg.rf.8744377541452bc0e6a20045dea52682.xml
      IMG_2383_jpg.rf.27c654b07f98a7d9986a79e5042d0a49.jpg
      IMG_2383_jpg.rf.27c654b07f98a7d9986a79e5042d0a49.xml
      IMG_2384_jpg.rf.41886e035cd3ce3f45ca15c4a3246077.jpg
      IMG_2384_jpg.rf.41886e035cd3ce3f45ca15c4a3246077.xml
      IMG_2385_jpg.rf.344180d5d58cebf7521015a7fffecfd7.jpg
      IMG_2385_jpg.rf.344180d5d58cebf7521015a7fffecfd7.xml
      IMG_2386_jpg.rf.8fe133e599d90ccff9413bca82e96025.jpg
      IMG_2386_jpg.rf.8fe1

In [ ]:
# Cell 3 - Verify
import os
voc_dir = '/content/dataset/voc/voc/Smartphone-Detection-2/train'
files = os.listdir(voc_dir)
imgs = [f for f in files if f.endswith('.jpg')]
xmls = [f for f in files if f.endswith('.xml')]
print(f"Images: {len(imgs)}")
print(f"XML annotations: {len(xmls)}")

Images: 157
XML annotations: 157


In [ ]:
!pip install torchvision -q

In [ ]:
import os
import shutil
import numpy as np
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
from torchvision import transforms

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_DIR  = "/content/dataset/voc/voc/Smartphone-Detection-2/train"
K_FOLDS      = 5
EPOCHS       = 50
IMG_SIZE     = 640
NUM_CLASSES  = 3  # background + person + smartphone
RESULTS_DIR  = "/content/results/fasterrcnn"
DEVICE       = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Using device: {DEVICE}")

# Fixed: Roboflow exports numeric labels in VOC format
CLASS_MAP   = {"0": 1, "1": 2}
LABEL_NAMES = {1: "person", 2: "smartphone"}

# ─── DATASET ──────────────────────────────────────────────────────────────────
class VOCDataset(Dataset):
    def __init__(self, image_paths, dataset_dir):
        self.image_paths = image_paths
        self.dataset_dir = Path(dataset_dir)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        xml_path = self.dataset_dir / (Path(img_path).stem + ".xml")

        img = Image.open(img_path).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE))
        img_tensor = transforms.ToTensor()(img)

        boxes, labels = [], []
        if xml_path.exists():
            tree = ET.parse(xml_path)
            root = tree.getroot()
            orig_w = int(root.find("size/width").text)
            orig_h = int(root.find("size/height").text)
            scale_x = IMG_SIZE / orig_w
            scale_y = IMG_SIZE / orig_h

            for obj in root.findall("object"):
                label = obj.find("name").text.strip()
                if label not in CLASS_MAP:
                    continue
                bbox = obj.find("bndbox")
                xmin = float(bbox.find("xmin").text) * scale_x
                ymin = float(bbox.find("ymin").text) * scale_y
                xmax = float(bbox.find("xmax").text) * scale_x
                ymax = float(bbox.find("ymax").text) * scale_y
                # Ensure valid box
                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(CLASS_MAP[label])

        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes":  boxes,
            "labels": labels,
        }
        return img_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

# ─── IoU HELPER ───────────────────────────────────────────────────────────────
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# ─── EVALUATION ───────────────────────────────────────────────────────────────
def evaluate(model, loader, iou_thresh=0.5):
    model.eval()
    all_tp, all_fp, all_fn = 0, 0, 0
    total_preds = 0
    total_gt    = 0

    with torch.no_grad():
        for images, targets in loader:
            images  = [img.to(DEVICE) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                pred_boxes  = output["boxes"].cpu().numpy()
                pred_scores = output["scores"].cpu().numpy()
                pred_labels = output["labels"].cpu().numpy()
                gt_boxes    = target["boxes"].numpy()
                gt_labels   = target["labels"].numpy()

                # Low threshold to catch all predictions
                keep        = pred_scores >= 0.05
                pred_boxes  = pred_boxes[keep]
                pred_labels = pred_labels[keep]
                pred_scores = pred_scores[keep]

                total_preds += len(pred_boxes)
                total_gt    += len(gt_boxes)

                if len(gt_boxes) == 0:
                    all_fp += len(pred_boxes)
                    continue

                if len(pred_boxes) == 0:
                    all_fn += len(gt_boxes)
                    continue

                # Sort by confidence descending
                order       = pred_scores.argsort()[::-1]
                pred_boxes  = pred_boxes[order]
                pred_labels = pred_labels[order]

                matched_gt = set()
                tp, fp = 0, 0

                for pb, pl in zip(pred_boxes, pred_labels):
                    best_iou = 0
                    best_j   = -1
                    for j, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                        if j in matched_gt:
                            continue
                        iou = compute_iou(pb, gb)
                        if iou > best_iou:
                            best_iou = iou
                            best_j   = j
                    if best_iou >= iou_thresh and best_j >= 0:
                        tp += 1
                        matched_gt.add(best_j)
                    else:
                        fp += 1

                fn = len(gt_boxes) - len(matched_gt)
                all_tp += tp
                all_fp += fp
                all_fn += fn

    print(f"  Debug — Preds: {total_preds} | GT boxes: {total_gt} | TP: {all_tp} | FP: {all_fp} | FN: {all_fn}")

    precision = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0
    recall    = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ─── MAIN TRAINING LOOP ───────────────────────────────────────────────────────
dataset_dir = Path(DATASET_DIR)
image_files = sorted([
    str(f) for f in dataset_dir.iterdir()
    if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
])
print(f"Total images found: {len(image_files)}")

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(image_files), 1):
    print(f"\n{'='*40}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"  Train: {len(train_idx)} images")
    print(f"  Test:  {len(val_idx)} images")
    print(f"{'='*40}")

    train_paths = [image_files[i] for i in train_idx]
    val_paths   = [image_files[i] for i in val_idx]

    train_dataset = VOCDataset(train_paths, dataset_dir)
    val_dataset   = VOCDataset(val_paths,   dataset_dir)

    train_loader = DataLoader(train_dataset, batch_size=4,
                              shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=4,
                              shuffle=False, collate_fn=collate_fn)

    # Load pretrained Faster R-CNN and replace head
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    model.to(DEVICE)

    optimizer = torch.optim.SGD(
        model.parameters(), lr=0.005,
        momentum=0.9, weight_decay=0.0005
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=10, gamma=0.1
    )

    # Training
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for images, targets in train_loader:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            print(f"  Epoch {epoch}/{EPOCHS} — Loss: {epoch_loss/len(train_loader):.4f}")

    # Evaluate
    precision, recall, f1 = evaluate(model, val_loader)
    fold_results.append({
        "fold":      fold,
        "precision": precision,
        "recall":    recall,
        "f1":        f1
    })

    print(f"\nFold {fold} Results:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1:        {f1:.4f}")

    # Save model
    torch.save(model.state_dict(),
               f"{RESULTS_DIR}/fasterrcnn_fold{fold}.pt")

# ─── FINAL SUMMARY ────────────────────────────────────────────────────────────
print(f"\n{'='*40}")
print("FASTER R-CNN K-FOLD FINAL RESULTS")
print(f"{'='*40}")
for r in fold_results:
    print(f"Fold {r['fold']}: Precision={r['precision']:.4f} | "
          f"Recall={r['recall']:.4f} | F1={r['f1']:.4f}")

means = {k: np.mean([r[k] for r in fold_results])
         for k in ["precision", "recall", "f1"]}
stds  = {k: np.std([r[k]  for r in fold_results])
         for k in ["precision", "recall", "f1"]}

print(f"\nMean ± Std across {K_FOLDS} folds:")
for k in ["precision", "recall", "f1"]:
    print(f"  {k}: {means[k]:.4f} ± {stds[k]:.4f}")

# Save to Drive
summary_path = "/content/drive/MyDrive/RD2/fasterrcnn_kfold_results.txt"
with open(summary_path, "w") as f:
    for r in fold_results:
        f.write(f"Fold {r['fold']}: Precision={r['precision']:.4f} | "
                f"Recall={r['recall']:.4f} | F1={r['f1']:.4f}\n")
    for k in ["precision", "recall", "f1"]:
        f.write(f"Mean {k}: {means[k]:.4f} ± {stds[k]:.4f}\n")

print(f"\nResults saved to Google Drive at RD2/fasterrcnn_kfold_results.txt")

Using device: cuda
Total images found: 157

FOLD 1/5
  Train: 125 images
  Test:  32 images
  Epoch 10/50 — Loss: 0.0446
  Epoch 20/50 — Loss: 0.0280
  Epoch 30/50 — Loss: 0.0274
  Epoch 40/50 — Loss: 0.0268
  Epoch 50/50 — Loss: 0.0274
  Debug — Preds: 76 | GT boxes: 64 | TP: 64 | FP: 12 | FN: 0

Fold 1 Results:
  Precision: 0.8421
  Recall:    1.0000
  F1:        0.9143

FOLD 2/5
  Train: 125 images
  Test:  32 images
  Epoch 10/50 — Loss: 0.0414
  Epoch 20/50 — Loss: 0.0286
  Epoch 30/50 — Loss: 0.0271
  Epoch 40/50 — Loss: 0.0275
  Epoch 50/50 — Loss: 0.0270
  Debug — Preds: 68 | GT boxes: 64 | TP: 64 | FP: 4 | FN: 0

Fold 2 Results:
  Precision: 0.9412
  Recall:    1.0000
  F1:        0.9697

FOLD 3/5
  Train: 126 images
  Test:  31 images
  Epoch 10/50 — Loss: 0.0439
  Epoch 20/50 — Loss: 0.0276
  Epoch 30/50 — Loss: 0.0265
  Epoch 40/50 — Loss: 0.0266
  Epoch 50/50 — Loss: 0.0264
  Debug — Preds: 69 | GT boxes: 62 | TP: 62 | FP: 7 | FN: 0

Fold 3 Results:
  Precision: 0.8986
  R

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader
from pathlib import Path
from PIL import Image
from torchvision import transforms
import xml.etree.ElementTree as ET

DATASET_DIR = "/content/dataset/voc/voc/Smartphone-Detection-2/train"
IMG_SIZE    = 640
NUM_CLASSES = 3
DEVICE      = torch.device("cuda")
CLASS_MAP   = {"0": 1, "1": 2}

dataset_dir = Path(DATASET_DIR)
image_files = sorted([
    str(f) for f in dataset_dir.iterdir()
    if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

# Load just 4 images for debugging
sample_paths = image_files[:4]

# Check what's in the XML files
print("=== CHECKING XML ANNOTATIONS ===")
for img_path in sample_paths:
    xml_path = dataset_dir / (Path(img_path).stem + ".xml")
    if xml_path.exists():
        tree = ET.parse(xml_path)
        root = tree.getroot()
        objects = root.findall("object")
        print(f"\n{Path(img_path).name}:")
        for obj in objects:
            name = obj.find("name").text
            bbox = obj.find("bndbox")
            print(f"  Label: '{name}' → mapped to: {CLASS_MAP.get(name.lower(), 'NOT FOUND')}")
    else:
        print(f"  No XML found for {Path(img_path).name}")

# Quick model test
print("\n=== CHECKING MODEL OUTPUT ===")
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)
model.eval()

img = Image.open(sample_paths[0]).convert("RGB")
img = img.resize((IMG_SIZE, IMG_SIZE))
img_tensor = transforms.ToTensor()(img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    output = model(img_tensor)

print(f"Number of detections: {len(output[0]['boxes'])}")
print(f"Scores: {output[0]['scores'][:10]}")
print(f"Labels: {output[0]['labels'][:10]}")

=== CHECKING XML ANNOTATIONS ===

IMG_2379_jpg.rf.7365fa9ff9cd1ff3a4cd55745336221f.jpg:
  Label: '0' → mapped to: 1
  Label: '1' → mapped to: 2

IMG_2380_jpg.rf.26f17b5795f5a201b3b39b9994431298.jpg:
  Label: '0' → mapped to: 1
  Label: '1' → mapped to: 2

IMG_2381_jpg.rf.0b7a47d4e8a67a988f15c32629ef3b49.jpg:
  Label: '0' → mapped to: 1
  Label: '1' → mapped to: 2

IMG_2382_jpg.rf.8744377541452bc0e6a20045dea52682.jpg:
  Label: '0' → mapped to: 1
  Label: '1' → mapped to: 2

=== CHECKING MODEL OUTPUT ===
Number of detections: 100
Scores: tensor([0.5834, 0.5667, 0.5644, 0.5458, 0.5399, 0.5363, 0.5335, 0.5306, 0.5284,
        0.5273], device='cuda:0')
Labels: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')


In [14]:
import shutil, os

os.makedirs('/content/drive/MyDrive/RD2/models/fasterrcnn', exist_ok=True)

for fold in range(1, 6):
    src = f'/content/results/fasterrcnn/fasterrcnn_fold{fold}.pt'
    dst = f'/content/drive/MyDrive/RD2/models/fasterrcnn/fasterrcnn_fold{fold}.pt'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✅ Saved fold {fold}')
    else:
        print(f'❌ Not found: {src}')

✅ Saved fold 1
✅ Saved fold 2
✅ Saved fold 3
✅ Saved fold 4
✅ Saved fold 5
